## ML_1043: Beam Search Decoding

**Difficulty**: Medium | **TorchCode**: #33

### Problem
Implement beam search decoding. Maintain `beam_width` hypotheses; at each step expand each beam with all tokens, keep the top-k by cumulative log-probability. Stop when EOS is reached or `max_len` is exceeded.

### Formula
$$\text{score}(y_{1:t}) = \sum_{i=1}^t \log P(y_i \mid y_{1:i-1})$$

In [ ]:
import torch

def beam_search(log_prob_fn, start_token, max_len, beam_width, eos_token):
    beams = [(0.0, [start_token])]
    completed = []
    for _ in range(max_len):
        candidates = []
        for score, seq in beams:
            if seq[-1] == eos_token:
                completed.append((score, seq))
                continue
            log_probs = log_prob_fn(torch.tensor(seq))
            topk_lp, topk_idx = log_probs.topk(beam_width)
            for j in range(beam_width):
                candidates.append((score + topk_lp[j].item(), seq + [topk_idx[j].item()]))
        if not candidates:
            break
        candidates.sort(key=lambda x: x[0], reverse=True)
        beams = candidates[:beam_width]
    all_seqs = completed + beams
    all_seqs.sort(key=lambda x: x[0], reverse=True)
    return all_seqs[0][1]

In [ ]:
import torch

# --- Test 1: Returns list starting with start_token ---
def dummy(tokens): return torch.zeros(10)
seq = beam_search(dummy, start_token=0, max_len=5, beam_width=3, eos_token=9)
assert isinstance(seq, list) and seq[0] == 0

# --- Test 2: Greedy path (beam=1) ---
def greedy_fn(tokens):
    lp = torch.full((5,), -10.0)
    lp[min(len(tokens), 4)] = 0.0
    return lp
seq = beam_search(greedy_fn, start_token=0, max_len=5, beam_width=1, eos_token=4)
assert seq == [0, 1, 2, 3, 4]

# --- Test 3: Beam finds better path than greedy ---
def tricky(tokens):
    lp = torch.full((6,), -100.0)
    if len(tokens) == 1:
        lp[1] = -1.0; lp[2] = -0.5
    elif tokens[-1] == 1:
        lp[5] = 0.0
    elif tokens[-1] == 2:
        lp[5] = -10.0
    else:
        lp[5] = 0.0
    return lp
seq = beam_search(tricky, start_token=0, max_len=5, beam_width=2, eos_token=5)
assert seq == [0, 1, 5]

# --- Test 4: Stops at EOS ---
def eos_fn(tokens):
    lp = torch.zeros(4); lp[3] = 10.0; return lp
seq = beam_search(eos_fn, start_token=0, max_len=100, beam_width=2, eos_token=3)
assert seq[-1] == 3 and len(seq) == 2

print('All tests passed!')